In [5]:
!pip install prophet
!pip install torchmetrics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 5.3 MB/s eta 0:00:0000:0100:01


In [9]:
import pandas as pd
import numpy as np
import torch
from prophet import Prophet
from torchmetrics import WeightedMeanAbsolutePercentageError
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [2]:
%cd /content/Walmart_sales_forecasting
data_dir = 'data/processed/feature_engineering.feather'
df_feature = pd.read_feather(data_dir)
df_feature

/content/Walmart_sales_forecasting


,Store,Dept,Date,Weekly_Sales,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,...,mean_sales_last_6_week,max_sales_last_6_week,min_sales_last_6_week,std_sales_last_6_week,emw_sales_0.5,emw_sales_0.75,sum_store_1_week,mean_store_1_week,sum_dept_1_week,mean_dept_1_week
6,1,1,2010-03-19,22136.64,A,151315,54.58,2.720,0.00,0.00,...,30360.018333,63593.12,7612.03,26384.953473,20940.430630,15246.700822,1242589.35,17258.185417,846686.47,18815.254889
149,1,2,2010-03-19,43615.49,A,151315,54.58,2.720,0.00,0.00,...,32780.786667,63593.12,7612.03,24477.831239,21538.535315,20414.155206,1242589.35,17258.185417,1742919.72,38731.549333
292,1,3,2010-03-19,9001.37,A,151315,54.58,2.720,0.00,0.00,...,29451.181667,61326.35,7612.03,20480.695326,32577.012657,37815.156301,1242589.35,17258.185417,371295.10,8251.002222
435,1,4,2010-03-19,34118.11,A,151315,54.58,2.720,0.00,0.00,...,20730.351667,43615.49,7612.03,14443.999382,20789.191329,16204.816575,1242589.35,17258.185417,1064848.12,23663.291556
578,1,5,2010-03-19,22632.57,A,151315,54.58,2.720,0.00,0.00,...,25148.031667,43615.49,9001.37,13661.565950,27453.650664,29639.786644,1242589.35,17258.185417,1063601.82,24734.926047
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
421012,45,93,2012-10-26,2487.80,B,118221,58.85,3.882,4018.91,58.08,...,17814.415000,54608.75,717.82,20287.128190,35354.118359,45312.322469,760281.43,11347.484030,1049693.34,24992.698571
421146,45,94,2012-10-26,5203.31,B,118221,58.85,3.882,4018.91,58.08,...,18109.411667,54608.75,1689.10,19999.636394,18920.959180,13193.930617,760281.43,11347.484030,1291378.41,29349.509318
421289,45,95,2012-10-26,56017.47,B,118221,58.85,3.882,4018.91,58.08,...,18695.113333,54608.75,2487.80,19466.945450,12062.134590,7200.965154,760281.43,11347.484030,1709432.51,37987.389111
421434,45,97,2012-10-26,6817.48,B,118221,58.85,3.882,4018.91,58.08,...,26666.748333,56017.47,2487.80,23647.747335,34039.802295,43813.343789,760281.43,11347.484030,596563.70,13558.265909


In [3]:
df_feature.info()

<class 'pandas.core.frame.DataFrame'>
Index: 401996 entries, 6 to 421569
Data columns (total 42 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   Store                   401996 non-null  int64         
 1   Dept                    401996 non-null  int64         
 2   Date                    401996 non-null  datetime64[ns]
 3   Weekly_Sales            401996 non-null  float64       
 4   Type                    401996 non-null  object        
 5   Size                    401996 non-null  int64         
 6   Temperature             401996 non-null  float64       
 7   Fuel_Price              401996 non-null  float64       
 8   MarkDown1               401996 non-null  float64       
 9   MarkDown2               401996 non-null  float64       
 10  MarkDown3               401996 non-null  float64       
 11  MarkDown4               401996 non-null  float64       
 12  MarkDown5               401996 non-

In [ ]:
test = df_feature[df_feature['is_test']]
train = df_feature[~df_feature['is_test']]

prophet_data = df_feature.groupby(['Date', 'store_dept']).agg(
    {
        'Weekly_Sales' : 'sum',
        'Size' : 'first',
        'Type' : 'first',
        'IsHoliday_True' : 'first',
        'Temperature': 'first',
        'Fuel_Price' : 'first',
        "total_markdown": "first",   
        "avg_markdown": "first",      
        "max_markdown": "first",
    }
).reset_index()

prophet_data

,Date,store_dept,Weekly_Sales,Size,Type,IsHoliday_True,Temperature,Fuel_Price
0,2010-03-19,store_10_dept_1,38252.33,126512,B,False,61.46,3.054
1,2010-03-19,store_10_dept_10,49479.06,126512,B,False,61.46,3.054
2,2010-03-19,store_10_dept_11,30518.98,126512,B,False,61.46,3.054
3,2010-03-19,store_10_dept_12,11762.04,126512,B,False,61.46,3.054
4,2010-03-19,store_10_dept_13,65575.03,126512,B,False,61.46,3.054
...,...,...,...,...,...,...,...,...
401991,2012-10-26,store_9_dept_91,914.84,125833,B,False,69.52,3.506
401992,2012-10-26,store_9_dept_92,18310.28,125833,B,False,69.52,3.506
401993,2012-10-26,store_9_dept_94,233.02,125833,B,False,69.52,3.506
401994,2012-10-26,store_9_dept_95,32382.05,125833,B,False,69.52,3.506


In [ ]:
def build_prophet_model(prophet_data,  test_data):
    min_test_time = test_data['Date'].min()
    max_test_time = test_data['Date'].max()
    
    prophet_model = {}
    prophet_predictions = {}
    prophet_metrics = pd.DataFrame(columns=['combo', 'mae', 'rmae', 'wape'])
    all_real = []
    all_predict = [] 
    
    
    for combination in prophet_data['store_dept'].unique():
        print(f'Build prophet model for {combination}')
        
        combo = prophet_data[prophet_data['store_dept'] == combination]
        combo = combo.rename(columns={'Date':'ds', 'Weekly_Sales' : 'y'}) #prophen require ds and y
        
        combo_train = combo[combo['ds'] < min_test_time]
        combo_test = combo[(combo['ds'] >= min_test_time) & (combo['ds'] <= max_test_time)]
        if not combo_train or combo_test:
            print(f'Skip {combination} due to lack of data')
            continue
        
        model = Prophet(daily_seasonality=False, monthly_seasonality=True, yearly_seasonality=True, seasonality_mode='multiplicative')
        
        for reg in ['Size', 'Type', 'IsHoliday_True', 'Temperature', 'Fuel_Price', 'total_markdown', 'avg_markdown', 'max_markdown']:
            model.add_regressor(reg)
        
        try:
            model.fit(combo_train)
        except Exception as e:
            print(f'Fail to train {combination} : {e}')
            continue
        
        # test
        future = combo_test[['ds', 'Size', 'Type', 'IsHoliday_True', 'Temperature', 'Fuel_Price', 'total_markdown', 'avg_markdown', 'max_markdown']]
        forecast = model.predict(future)
        
        forecast = forecast[['ds', 'yhat', 'yhat_upper', 'yhat_lower']].merge(combo_test[['ds', 'y']], on=['ds'])
        
        prophet_model[combination] = model
        prophet_predictions[combination] = forecast
        
        # evaluate
        mae = mean_absolute_error(forecast['y'], forecast['yhat'])
        rmse = np.sqrt(mean_squared_error(forecast['y'], forecast['yhat']))
        wape = WeightedMeanAbsolutePercentageError(torch.from_numpy(np.array(forecast['y'])), torch.from_numpy(np.array(forecast['yhat'])))
        
        prophet_metrics[len(prophet_metrics)] = [combo, mae, rmse, wape]
        
        #overall evaluation
        all_real.extend(forecast['y'])
        all_predict.extend(forecast['yhat'])
        
    
    mean_mae = np.mean(prophet_metrics['mae'])
    mean_rmse = np.mean(prophet_metrics['rmse'])
    ovr_wape = WeightedMeanAbsolutePercentageError(torch.from_numpy(np.array(all_real)), torch.from_numpy(np.array(all_predict)))
    
    return prophet_model, prophet_predictions, (mean_mae, mean_rmse, ovr_wape)


prophet_model, prophet_predictions, (mean_mae, mean_rmse, ovr_wape) = build_prophet_model(prophet_data,  test)
    
        
        
        
        
        
        
        
        
        
        
    
    